# Counting objects with StarDist workflow

---
*Introduction to Image Analysis Workshop*

*Stefania Marcotti (stefania.marcotti@crick.ac.uk) and Dave Barry (david.barry@crick.ac.uk)*

*Intro to detecting objects with StarDist*

*CC-BY-SA-4.0 license: creativecommons.org/licenses/by-sa/4.0/*

---

In this notebook, we build on earlier work to show how StarDist can be used to count objects in microscopy images.

<img width="20%" src="https://raw.githubusercontent.com/stardist/stardist/main/images/stardist_logo.jpg" alt="Stardist logo">

### Import libraries
In addition to the libaries installed previoulsy, you will also need to install stardist to run this notebook - to do so, follow the instructions on the [StarDist project page](https://github.com/stardist/stardist/#installation).

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
Install <code>stardist</code> in the workshop environment. To do so, follow the instructions at the link above. Please note, the CPU release of <code>tensorflow</code> should be sufficient to run this notebook. If the cell below runs without errors or warnings, the installation has been successful. </div>

In [ ]:
import numpy as np

from skimage import io
from skimage import measure

from stardist.models import StarDist2D
from csbdeep.utils import normalize

import matplotlib.pyplot as plt

import pandas as pd

### Open images

To get started with our "counting objects" task, the first thing we need to do is to open an image. We will use as example image [`003003-10.tif`](https://github.com/RMS-DAIM/introduction-to-image-analysis/blob/main/Data/idr0028/003003-10.tif), where some cells have been stained for *nuclei (Hoescht, ch0)*, *tubulin (AlexaFluor568, ch1)*, *actin (AlexaFluor488, ch2)*, and *YAP/TAZ (AlexaFluor647, ch3)* and imaged with fluorescent microscopy. You can find more information about this [public image dataset](https://idr.openmicroscopy.org/webclient/?show=plate-5679) in [the associated publication](https://www.nature.com/articles/sdata201718).

Next, we will display our example image to screen - it's always good practice to get an idea about the kind of data we're working with! We will be working only with the nuclei (ch0), so we will need to make sure to select the correct channel.

In [ ]:
# read .tif file
im_read = io.imread('../../Data/idr0028/003003-10.tif')

In [ ]:
# print image type
print('Image type:', type(im_read))

In [ ]:
# print image dimensions
print('Image dimensions:', im_read.shape)

The image dimensions are currently in the (y, x, c) format. Ideally, by convention, we would want the dimensions of the image to be (c, x, y). We can rearrange the dimensions by using the numpy function `transpose`.

In [ ]:
# rearrange channels
im_all = np.transpose(im_read, (2, 0, 1))
print('New image dimensions:', im_all.shape)

In [ ]:
# display all channels 
fig, axs = plt.subplots(2, 2, figsize=(8,6))

axs[0, 0].imshow(im_all[0,], cmap='gray')
axs[0, 0].set_title('ch0: nuclei (Hoechst)')
axs[0, 0].axis('off')

axs[0, 1].imshow(im_all[1,], cmap='gray')
axs[0, 1].set_title('ch1: tubulin (AlexaFluor568)')
axs[0, 1].axis('off')

axs[1, 0].imshow(im_all[2,], cmap='gray')
axs[1, 0].set_title('ch2: actin (AlexaFluor488)')
axs[1, 0].axis('off')

axs[1, 1].imshow(im_all[3,], cmap='gray')
axs[1, 1].set_title('ch3: YAP/TAZ (AlexaFluor647)')
axs[1, 1].axis('off')

plt.tight_layout()

In [ ]:
# select the channel containing the nuclei
im = im_all[0,]

In [ ]:
# display image
fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(im, cmap='gray')
ax.set_title('ch0: nuclei (Hoechst)')
ax.axis('off')
plt.tight_layout()

### Segment and count objects using StarDist

The next step in our pipeline is segmentation. In earlier notebooks, we used grey level thresholding for this task, but here, we will use StarDist.

In [ ]:
model_name = '2D_versatile_fluo'
print(f"Loading StarDist model: {model_name}")
model = StarDist2D.from_pretrained(model_name)

# Normalize image (StarDist expects normalized input)
img_normalized = normalize(im, 1, 99.8)

# Run StarDist prediction
labels, details = model.predict_instances(img_normalized)

print(f"  -> Detected {labels.max()} objects")

# display the binary mask
fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(labels, cmap='jet')
ax.set_title('Nuclear Labels')
ax.axis('off')
plt.tight_layout()

### Perform basic morphological quantification
We can perform some basic morphological quantification by using the function [`skimage.measure.regionprops_table`](https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops_table). Check the documentation to see all the different output we can obtain!

In this example, we are going to show the area, centroid (y and x coordinates), and eccentricity for each blob. We are going to save the data in a [Pandas dataframe](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html), which is a data structure containing labelled axes (one row for each object, one column for each feature).

In [ ]:
# measure properties
props = measure.regionprops_table(labels, im, properties=['area', 'centroid', 'eccentricity'])
props_df = pd.DataFrame(props)

props_df.head(5)

In [ ]:
# display some results
fig, axs = plt.subplots(1, 2, figsize=(4,3))

axs[0].boxplot(props_df['area'])
axs[0].set_title('Nuclear area (px)')

axs[1].boxplot(props_df['eccentricity'])
axs[1].set_title('Nuclear eccentricity')

plt.tight_layout()